# Chapter 4 — Disks and a Milky Way potential

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/04_disks_milky_way.ipynb)

**Goal.** Build a gravitational potential for the Milky Way out of a bulge, a disk and a dark-matter
halo, and compare its rotation curve with observations.

**You will learn**
1. cylindrical coordinates $(R, \phi, z)$, the natural coordinates for a disk galaxy;
2. why the circular speed of a disk is *not* $\sqrt{GM(<R)/R}$, using the exponential disk;
3. the Miyamoto–Nagai disk, an analytic flattened potential;
4. how AGAMA combines components into one potential, and how to load published Milky Way models;
5. why the Milky Way needs a dark-matter halo.

**Units.** From this chapter on we work in physical units: masses in $M_\odot$, lengths in kpc,
velocities in km/s (Chapter 0 §0.2). Following AGAMA's rule, `setUnits` is the first thing we call,
before creating any AGAMA object.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # 1 Msun, 1 kpc, 1 km/s -- before any AGAMA object!
G = agama.G                                        # 4.30e-6 kpc (km/s)^2 / Msun

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0, i1, k0, k1           # modified Bessel functions I_0, I_1, K_0, K_1
from scipy.optimize import minimize_scalar

def vcirc(pot, R):
    """Circular speed in the midplane, v_c = sqrt(-R F_R), at radii R (see Chapter 0 §0.4)."""
    R = np.atleast_1d(R)                           # accept a single number or an array
    return np.sqrt(-R * pot.force(np.column_stack([R, 0 * R, 0 * R]))[:, 0])

## 4.1 Cylindrical coordinates

A disk galaxy is (nearly) axisymmetric: it looks the same after any rotation about its symmetry
axis, which we take as the $z$-axis. The natural coordinates are cylindrical:

$$ R = \sqrt{x^2 + y^2}, \qquad \phi = \arctan(y/x), \qquad z, $$

with velocity components

$$ v_R = \frac{x v_x + y v_y}{R}, \qquad v_\phi = \frac{x v_y - y v_x}{R}, \qquad v_z . $$

where $(x, y, z)$ are Cartesian positions and $(v_x, v_y, v_z)$ the Cartesian velocity components (the arctangent is
taken in the correct quadrant, `np.arctan2(y, x)`). $v_R$ is the speed away from the axis, $v_\phi$ the rotation speed (positive for counter-clockwise
rotation seen from $+z$), and $R\,v_\phi = x v_y - y v_x$ is the $z$-component of angular momentum,
$L_z$, which is conserved in an axisymmetric potential (Chapter 5).

AGAMA works in Cartesian coordinates: positions and velocities are `(N, 6)` arrays
$(x, y, z, v_x, v_y, v_z)$. We therefore write our own converters, and check them with a round trip.

In [ ]:
def cart_to_cyl(xv):
    """(N, 6) Cartesian (x, y, z, vx, vy, vz) -> (N, 6) cylindrical (R, phi, z, vR, vphi, vz)."""
    x, y, z, vx, vy, vz = xv.T                     # .T makes each of the 6 columns a separate array
    R = np.hypot(x, y)                             # sqrt(x^2 + y^2), computed without overflow
    phi = np.arctan2(y, x)                         # angle in (-pi, pi], correct in all four quadrants
    return np.column_stack([R, phi, z, (x * vx + y * vy) / R, (x * vy - y * vx) / R, vz])

def cyl_to_cart(rv):
    """Inverse of cart_to_cyl."""
    R, phi, z, vR, vphi, vz = rv.T
    c, s = np.cos(phi), np.sin(phi)
    return np.column_stack([R * c, R * s, z, vR * c - vphi * s, vR * s + vphi * c, vz])

rng = np.random.default_rng(42)                    # fixed seed: the same "random" numbers every run
xv = rng.normal(0, 10, size=(1000, 6))             # 1000 random positions [kpc] and velocities
roundtrip_err = np.max(np.abs(cyl_to_cart(cart_to_cyl(xv)) - xv))
print(f'max error of the round trip Cartesian -> cylindrical -> Cartesian: {roundtrip_err:.1e}')
assert roundtrip_err < 1e-12

## 4.2 Disks are different: the exponential disk

The surface density of most galaxy disks falls off exponentially with radius (Freeman 1970):

$$ \Sigma(R) = \Sigma_0\, e^{-R/R_d}, \qquad M_d = 2\pi\Sigma_0 R_d^2 , $$

where $\Sigma$ is the mass per unit area of the disk, $\Sigma_0$ its central value, $R_d$ the **scale length** and $M_d$ the total
mass. For a *razor-thin* disk Freeman showed
that the circular speed in the plane is

$$ v_c^2(R) = 4\pi G \Sigma_0 R_d\, y^2 \left[I_0(y) K_0(y) - I_1(y) K_1(y)\right],
   \qquad y = \frac{R}{2R_d}, $$

where $v_c(R)$ is the circular speed (the speed of a circular orbit at radius $R$), $G$ the gravitational constant, and $I_n$ and
$K_n$ the modified Bessel functions of the first and second kind of order $n$ (available in `scipy.special`). It peaks at
$R \approx 2.15\,R_d$.

In a sphere, the shell theorem gives $v_c^2 = GM(<R)/R$. A disk has no shell theorem: mass *outside*
radius $R$ pulls a star at $R$ outwards, and the near side of the disk pulls harder than the far side.
To see the difference, compare with a sphere that has the same enclosed mass as the disk at every
radius, $M(<R) = M_d\,[1 - (1 + R/R_d)\,e^{-R/R_d}]$, where $M(<R)$ is the disk mass inside radius $R$.

*In AGAMA*, `type='Disk'` is an exponential disk with `surfaceDensity` $\Sigma_0$, `scaleRadius`
$R_d$ and `scaleHeight` $h$ (0 for razor-thin). AGAMA computes its potential approximately — an
analytic "disk ansatz" plus a numerical correction (a Multipole expansion, Chapter 1 §1.5) — so the
Freeman formula is an independent check.

In [ ]:
Md, Rd = 5e10, 3.0                                 # a Milky-Way-like disk: 5e10 Msun, 3 kpc
Sigma0 = Md / (2 * np.pi * Rd**2)                  # central surface density [Msun / kpc^2]

def vc_freeman(R):
    y = R / (2 * Rd)
    return np.sqrt(4 * np.pi * G * Sigma0 * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y)))

def vc_same_mass_sphere(R):
    M_enc = Md * (1 - (1 + R / Rd) * np.exp(-R / Rd))
    return np.sqrt(G * M_enc / R)

disk_thin = agama.Potential(type='Disk', surfaceDensity=Sigma0, scaleRadius=Rd, scaleHeight=0)
print(disk_thin)                                   # shows AGAMA's internal representation

R = np.linspace(0.1, 30, 300)
plt.figure(figsize=(7, 4))
plt.plot(R, vc_freeman(R), 'k-', lw=3, alpha=0.3, label='thin disk, Freeman formula')
plt.plot(R, vcirc(disk_thin, R), 'r--', label='thin disk, AGAMA')
plt.plot(R, vc_same_mass_sphere(R), 'b:', label='sphere with the same $M(<R)$')
plt.xlabel('$R$ [kpc]'); plt.ylabel('$v_c$ [km/s]'); plt.legend(); plt.show()

err = np.max(np.abs(vcirc(disk_thin, R) / vc_freeman(R) - 1))
R_peak = minimize_scalar(lambda r: -vc_freeman(r), bounds=(1, 15), method='bounded').x
excess = vc_freeman(R_peak) / vc_same_mass_sphere(R_peak) - 1
print(f'AGAMA vs Freeman: max relative difference {err:.1e}')
print(f'peak of the disk rotation curve at R = {R_peak / Rd:.3f} R_d; '
      f'there the disk spins {100 * excess:.0f}% faster than the same-mass sphere')
assert err < 2e-3 and abs(R_peak / Rd - 2.15) < 0.01

**Figure 4.1.** Midplane circular speed $v_c(R)$ of a razor-thin exponential disk ($M_d = 5\times10^{10}\,M_\odot$, $R_d = 3$ kpc): Freeman's formula (thick grey), AGAMA's `Disk` potential (red dashed), and a sphere with the same enclosed mass $M(<R)$ (blue dotted).

In Figure 4.1, inside about $R_d$ the sphere is faster: there, the disk mass outside $R$ pulls the star *outwards*,
partly cancelling the inward pull. Beyond that the disk is faster — at its peak by 15% — because the
disk material just inside $R$ is concentrated close to the star, in the plane, and pulls harder than
the same mass spread over a sphere. Far out, where almost all the mass is enclosed, the two curves
converge. AGAMA's approximation agrees with Freeman's formula to about $10^{-3}$, adequate for any
practical purpose.

Even the exponential disk has a falling rotation curve beyond $2.15\,R_d$. A disk whose rotation
curve is exactly flat needs $\Sigma \propto 1/R$ (the Mestel disk; Mestel 1963) — far more mass at
large radius than the light of galaxies shows. Keep this in mind for §4.5.

## 4.3 A disk with thickness: Miyamoto–Nagai

Real disks have a thickness. The **Miyamoto–Nagai** (1975) potential is a simple analytic model of
a thick disk:

$$ \Phi(R, z) = -\frac{GM}{\sqrt{R^2 + \left(a + \sqrt{z^2 + b^2}\right)^2}} . $$

Here $\Phi$ is the gravitational potential, $M$ the total mass, and $a$ and $b$ two lengths: $a$ sets the radial extent and $b$ the thickness. For $b \to 0$ it becomes a razor-thin disk (the
Kuzmin disk); for $a \to 0$ it becomes the Plummer sphere of Chapter 1. Its midplane circular speed is
$v_c^2 = GMR^2 / [R^2 + (a+b)^2]^{3/2}$.

AGAMA's `type='MiyamotoNagai'` takes `mass`, `scaleRadius` ($a$) and `scaleHeight` ($b$). Besides
the potential and force, any AGAMA potential returns its density $\rho = \nabla^2\Phi / 4\pi G$ via
`pot.density(points)`, which we use to draw density contours in the $(R, z)$ plane.

In [ ]:
M_mn, a_mn = 6.8e10, 3.0
b_values = [0.28, 1.0, 3.0]                        # thin, medium and thick versions

Rg, zg = np.meshgrid(np.linspace(0, 15, 151), np.linspace(-6, 6, 121))   # 2D grids of R and z values
points = np.column_stack([Rg.ravel(), 0 * Rg.ravel(), zg.ravel()])      # ravel() flattens a 2D grid to 1D

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for i, b_mn in enumerate(b_values):                # enumerate gives (index, value) pairs
    mn = agama.Potential(type='MiyamotoNagai', mass=M_mn, scaleRadius=a_mn, scaleHeight=b_mn)
    rho = mn.density(points).reshape(Rg.shape)     # back to the 2D grid shape for plotting
    Phi = mn.potential(points).reshape(Rg.shape)
    ax[i].contour(Rg, zg, np.log10(rho), levels=np.arange(4, 10.5, 0.5), cmap='viridis')
    ax[i].contour(Rg, zg, Phi, levels=8, colors='gray', linestyles='--', linewidths=0.8)
    ax[i].set_aspect('equal'); ax[i].set_xlabel('$R$ [kpc]'); ax[i].set_ylabel('$z$ [kpc]')
    ax[i].set_title(f'$a = {a_mn}$, $b = {b_mn}$ kpc')
plt.suptitle(r'Miyamoto–Nagai: density (colour, $\log_{10}\rho$ every 0.5 dex) and potential (grey dashed)')
plt.tight_layout(); plt.show()

# check AGAMA against the analytic midplane circular speed and the vertical force
mn = agama.Potential(type='MiyamotoNagai', mass=M_mn, scaleRadius=a_mn, scaleHeight=0.28)
R = np.linspace(0.1, 30, 100)
vc_mn = np.sqrt(G * M_mn * R**2 / (R**2 + (a_mn + 0.28)**2)**1.5)
# vertical force F_z = -dPhi/dz = -G M z (a + zeta) / [zeta (R^2 + (a + zeta)^2)^{3/2}], zeta = sqrt(z^2 + b^2)
Rz, z = 8.0, np.linspace(-3, 3, 61)
zeta = np.sqrt(z**2 + 0.28**2)
Fz_exact = -G * M_mn * z * (a_mn + zeta) / (zeta * (Rz**2 + (a_mn + zeta)**2)**1.5)
Fz_agama = mn.force(np.column_stack([Rz + 0 * z, 0 * z, z]))[:, 2]
print(f'max relative error: v_c {np.max(np.abs(vcirc(mn, R) / vc_mn - 1)):.1e}, '
      f'F_z {np.max(np.abs(Fz_agama - Fz_exact)) / np.max(np.abs(Fz_exact)):.1e}')
assert np.allclose(vcirc(mn, R), vc_mn, rtol=1e-10) and np.allclose(Fz_agama, Fz_exact, rtol=1e-10, atol=1e-12)

**Figure 4.2.** Miyamoto–Nagai models with $a = 3$ kpc and $b = 0.28$, 1 and 3 kpc (left to right) in the meridional $(R, z)$ plane: density contours ($\log_{10}\rho$ every 0.5 dex, colour) and equipotentials (grey dashed).

In Figure 4.2, small $b$ gives a thin, flattened density; large $b$ an almost round one. The potential contours
(grey) are much rounder than the density contours — a general property: the potential at any point
sums the pull of the whole galaxy, which smooths out the shape of the mass distribution. Note also the unrealistic feature of this model: the density falls off only as a power
law with height, not exponentially like real disks.

## 4.4 Building a Milky Way: composite potentials

A galaxy model is a sum of components. Because Poisson's equation is linear, potentials and forces
simply add, so the circular speeds add *in quadrature*:

$$ \Phi = \sum_i \Phi_i, \qquad v_c^2 = R\,\frac{\partial\Phi}{\partial R} = \sum_i v_{c,i}^2 . $$

where $\Phi_i$ is the potential of component $i$ and $v_{c,i}^2 = R\,\partial\Phi_i/\partial R$ its contribution to the circular speed, both in the midplane.

A widely used simple model is **MWPotential2014** (Bovy 2015), with
- a **bulge**: a power-law density $\rho \propto r^{-1.8}$ with an exponential cutoff at 1.9 kpc;
- a **disk**: Miyamoto–Nagai with $a = 3$ kpc, $b = 0.28$ kpc, $M = 6.8\times10^{10}\,M_\odot$;
- a **dark-matter halo**: an NFW profile (Navarro, Frenk & White 1996),
  $\rho \propto (r/r_s)^{-1}(1 + r/r_s)^{-2}$, with $r_s = 16$ kpc.

Here $\rho$ is the mass density, $r = \sqrt{R^2 + z^2}$ the spherical radius and $r_s$ the halo's scale radius.

AGAMA's `type='Spheroid'` covers both the bulge and the halo: it is the double power law
$\rho = \rho_0 (r/a)^{-\gamma}\,[1 + (r/a)^\alpha]^{(\gamma-\beta)/\alpha} \exp[-(r/r_{\rm cut})^\xi]$, with
normalisation $\rho_0$ (`densityNorm`), scale radius $a$ (`scaleRadius`), inner slope $\gamma$ (`gamma`), outer slope $\beta$ (`beta`),
$\alpha$ (`alpha`) setting how sharply the slope changes around $r = a$, and an optional cutoff at $r_{\rm cut}$ (`outerCutoffRadius`)
of strength $\xi$ (`cutoffStrength`). NFW is
$\gamma = 1$, $\beta = 3$. Passing several `Potential` objects to `agama.Potential(...)` combines them;
`pot[i]` retrieves component `i`.

In [ ]:
bulge = agama.Potential(type='Spheroid', densityNorm=2.22694e8, gamma=1.8, beta=1.8,
                        scaleRadius=1, outerCutoffRadius=1.9)
disk = agama.Potential(type='MiyamotoNagai', mass=6.819386e10, scaleRadius=3.0, scaleHeight=0.28)
halo = agama.Potential(type='Spheroid', densityNorm=8.48683e6, gamma=1, beta=3, scaleRadius=16)
mw2014 = agama.Potential(bulge, disk, halo)        # one composite potential from three components
print(mw2014)

R = np.linspace(0.2, 30, 300)
vc2_sum = vcirc(bulge, R)**2 + vcirc(disk, R)**2 + vcirc(halo, R)**2
print(f'v_c^2 of the composite vs sum of components: max relative difference '
      f'{np.max(np.abs(vcirc(mw2014, R)**2 / vc2_sum - 1)):.1e}')
assert np.allclose(vcirc(mw2014, R)**2, vc2_sum, rtol=1e-12)

**Published models shipped with AGAMA.** AGAMA comes with parameter files (`.ini`) for several Milky
Way models in its `data` folder, including MWPotential2014 and the more detailed model of
McMillan (2017), which has thin and thick stellar disks, two gas disks, a flattened bulge and an NFW
halo. `agama.Potential(filename)` loads one. We check that our hand-built MWPotential2014 matches the
file version, and that each model has the circular speed at the Sun's radius that its authors quote:
MWPotential2014 is normalised to 220 km/s at $R = 8$ kpc; McMillan's best-fitting model has the Sun at
$R_0 = 8.21$ kpc, and his paper estimates $v_0 = 232.8 \pm 3.0$ km/s.

In [ ]:
data_dir = os.path.join(os.path.dirname(agama.__file__), 'data')   # where AGAMA's .ini files live
mw2014_file = agama.Potential(os.path.join(data_dir, 'MWPotential2014.ini'))
mcmillan = agama.Potential(os.path.join(data_dir, 'McMillan17.ini'))

diff = np.max(np.abs(vcirc(mw2014, R) / vcirc(mw2014_file, R) - 1))
vc_2014 = vcirc(mw2014, 8.0)[0]
vc_mcm = vcirc(mcmillan, 8.21)[0]
print(f'our MWPotential2014 vs the .ini file: max relative difference in v_c {diff:.1e}')
print(f'v_c(8 kpc) in MWPotential2014 = {vc_2014:.2f} km/s;  v_c(8.21 kpc) in McMillan17 = {vc_mcm:.2f} km/s')
assert diff < 1e-4 and abs(vc_2014 - 220) < 0.5 and abs(vc_mcm - 232.8) < 3.0

The tiny difference between our MWPotential2014 and the file version comes from AGAMA's numerical
representation of the `Spheroid` components (a Multipole expansion on a grid, set up slightly
differently in the two cases).

## 4.5 The rotation curve and dark matter

Now compare with the measured rotation curve of the Milky Way. Eilers et al. (2019) used the
positions and velocities of 23,000 red-giant stars (APOGEE, WISE, 2MASS and Gaia) to measure

$$ v_c(R) = 229.0\ {\rm km/s} - 1.7\ {\rm km\,s^{-1}\,kpc^{-1}}\,(R - R_\odot), \qquad 5 \le R \le 25\ {\rm kpc}, $$

with $R_\odot = 8.122$ kpc, and systematic uncertainties of a few per cent. We plot it in Figure 4.3 as a band
of $\pm 3\%$.

In [ ]:
R0_eilers = 8.122
R_obs = np.linspace(5, 25, 50)
vc_obs = 229.0 - 1.7 * (R_obs - R0_eilers)

R = np.linspace(0.2, 30, 300)
vc_baryons = np.sqrt(vcirc(bulge, R)**2 + vcirc(disk, R)**2)   # stars only, no dark matter

plt.figure(figsize=(8, 5))
plt.fill_between(R_obs, 0.97 * vc_obs, 1.03 * vc_obs, color='gray', alpha=0.3,
                 label='measured (Eilers et al. 2019), $\\pm3\\%$')
plt.plot(R, vcirc(bulge, R), ':', label='bulge')
plt.plot(R, vcirc(disk, R), '--', label='disk')
plt.plot(R, vcirc(halo, R), '-.', label='dark halo')
plt.plot(R, vc_baryons, color='C4', lw=1.5, label='bulge + disk (no dark matter)')
plt.plot(R, vcirc(mw2014, R), 'k-', lw=2, label='MWPotential2014 total')
plt.plot(R, vcirc(mcmillan, R), 'C3-', lw=1, label='McMillan (2017) total')
plt.xlabel('$R$ [kpc]'); plt.ylabel('$v_c$ [km/s]'); plt.ylim(0, 300); plt.legend(fontsize=9); plt.show()

i8 = np.argmin(np.abs(R - 8.0))                    # index of the grid point closest to R = 8 kpc
for name, comp in [('bulge', bulge), ('disk', disk), ('halo', halo)]:
    print(f'fraction of v_c^2 at R = 8 kpc from the {name}: {vcirc(comp, 8.0)[0]**2 / vc_2014**2:.2f}')
v20_bar = np.sqrt(vcirc(bulge, 20.0)[0]**2 + vcirc(disk, 20.0)[0]**2)
v20_obs = 229.0 - 1.7 * (20 - R0_eilers)
print(f'at R = 20 kpc: stars alone give {v20_bar:.0f} km/s; observed {v20_obs:.0f} km/s')
for Rx in [10.0, 15.0, 25.0]:
    vobs = 229.0 - 1.7 * (Rx - R0_eilers)
    print(f'R = {Rx:4.0f} kpc: (model / observed - 1) = {vcirc(mw2014, Rx)[0] / vobs - 1:+.3f} (MWPotential2014), '
          f'{vcirc(mcmillan, Rx)[0] / vobs - 1:+.3f} (McMillan 2017)')
assert v20_bar < 0.8 * v20_obs

**Figure 4.3.** Rotation curve of MWPotential2014 split into bulge (dotted), disk (dashed) and dark halo (dash-dotted), with bulge + disk alone (purple) and the total (black); red: the total of McMillan's (2017) model; grey band: the Eilers et al. (2019) measurement $\pm 3\%$.

The stars alone (bulge + disk; Figure 4.3) produce a rotation curve that peaks near 5 kpc and then falls,
roughly as a point mass would far out. The observed curve stays nearly flat to 25 kpc. The
difference is attributed to a **dark-matter halo**: in MWPotential2014 it supplies about a third of
$v_c^2$ at the Sun's radius and dominates further out. (MOND and other modified-gravity theories offer
an alternative explanation; the standard cosmological model explains it with dark matter.)

Neither model matches the measurement perfectly. MWPotential2014 lies 4–6% below it between 10 and
25 kpc — its halo is a little too light for these data — while McMillan (2017), fitted to earlier
constraints, lies 3–10% above, increasingly so at large radius. Both models predate the Eilers et al.
measurement; Exercise 4 asks you to adjust the halo.

### How much mass?

For a sphere, the mass inside radius $r$ is $M(<r) = v_c^2 r / G$. For a flattened galaxy this is only an estimate. AGAMA's
`enclosedMass` integrates the actual density within a sphere of radius $r$, so we can test the
estimate.

In [ ]:
for r in [8.0, 20.0, 50.0]:
    M_true = mw2014.enclosedMass(r)
    M_est = vcirc(mw2014, r)[0]**2 * r / G
    print(f'r = {r:4.0f} kpc:  M(<r) = {M_true:.3e} Msun;  v_c^2 r / G = {M_est:.3e} Msun  '
          f'(ratio {M_est / M_true:.3f})')
    assert abs(M_est / M_true - 1) < 0.2

The spherical estimate overestimates the mass by 12% at the Sun, where the disk matters most, and
improves further out (4% at 20 kpc, 1% at 50 kpc), where the round halo dominates. Within 20 kpc this
Milky Way model contains about $1.7\times10^{11}\,M_\odot$, of which the stars are $7\times10^{10}\,M_\odot$:
most of it is dark.

## Summary

- Disk galaxies are described in cylindrical coordinates $(R, \phi, z)$.
- A disk's circular speed is not $\sqrt{GM(<R)/R}$: an exponential disk peaks at $2.15\,R_d$
  and spins faster there than a sphere of the same enclosed mass.
- Miyamoto–Nagai gives an analytic thick disk; AGAMA's `Disk` type gives a realistic exponential disk.
- Components combine by adding potentials, so circular speeds add in quadrature;
  `agama.Potential(c1, c2, c3)` builds a composite, `agama.Potential('file.ini')` loads a published model.
- The stars alone cannot explain the flat rotation curve of the Milky Way: a dark halo is needed.

| task | AGAMA |
|---|---|
| exponential disk | `agama.Potential(type='Disk', surfaceDensity=..., scaleRadius=..., scaleHeight=...)` |
| Miyamoto–Nagai disk | `type='MiyamotoNagai', mass=..., scaleRadius=a, scaleHeight=b` |
| bulge, NFW halo | `type='Spheroid', densityNorm=..., gamma=..., beta=..., scaleRadius=...` |
| combine components | `agama.Potential(pot1, pot2, ...)`; component `i` is `pot[i]` |
| published Milky Way models | `agama.Potential(os.path.join(data_dir, 'McMillan17.ini'))` |
| density of any potential | `pot.density(points)` |

## Exercises

1. **Thickness.** Repeat §4.2 for `scaleHeight` = 0.3 and 1 kpc. How does thickness change the peak
   circular speed? Why?
2. **Mestel disk.** Show (by thinking, then numerically with an AGAMA `Disk` of large `scaleRadius`)
   that $\Sigma \propto 1/R$ gives a flat rotation curve. What is the problem with its total mass?
3. **Kuzmin limit.** Set `scaleHeight` of the Miyamoto–Nagai disk to $10^{-3}$ kpc and plot the
   vertical force $F_z$ at $R = 8$ kpc against $z$. Explain the jump at $z = 0$ (Gauss's law gives
   $|F_z| = 2\pi G\Sigma$ just above a thin sheet).
4. **Halo mass.** Scale the halo `densityNorm` of MWPotential2014 by 0.5 and 1.5. Which version is
   closest to the Eilers et al. curve between 10 and 25 kpc?
5. **Two models.** Plot the bulge, disk and halo contributions of McMillan (2017) by building its
   components from `McMillan17.ini` (open the file to read the parameters), and compare the dark-matter
   fraction at the Sun with MWPotential2014.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §2.3 and §2.6.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapters 7–8: https://galaxiesbook.org
- K. C. Freeman (1970), ApJ 160, 811: https://ui.adsabs.harvard.edu/abs/1970ApJ...160..811F
- L. Mestel (1963), MNRAS 126, 553: https://ui.adsabs.harvard.edu/abs/1963MNRAS.126..553M
- M. Miyamoto & R. Nagai (1975), PASJ 27, 533: https://ui.adsabs.harvard.edu/abs/1975PASJ...27..533M
- J. Bovy (2015), "galpy: a Python library for galactic dynamics", ApJS 216, 29:
  https://ui.adsabs.harvard.edu/abs/2015ApJS..216...29B
- P. J. McMillan (2017), MNRAS 465, 76: https://doi.org/10.1093/mnras/stw2759
- A.-C. Eilers et al. (2019), ApJ 871, 120: https://doi.org/10.3847/1538-4357/aaf648
- J. F. Navarro, C. S. Frenk & S. D. M. White (1996), ApJ 462, 563:
  https://ui.adsabs.harvard.edu/abs/1996ApJ...462..563N